# Progress presentation — QR / CQR quantile load bounds

**Setup:** `conda activate pjm`, then open this notebook from the repo root (or run the first code cell to `cd` there).

Saved runs (under `results/`):
- **qr_lr5.0** — plain QR at τ=0.95 (strong year-average ECE)
- **cqr_lr5.0_og** — conformal adjustment (can over-cover)
- **winter0.25** — CQR with winter offset weighted by 0.25

The Python module [`plot_progress_presentation.py`](plot_progress_presentation.py) implements all plotting; this notebook regenerates figures and shows them inline.

Outputs in `results/figures_progress/`: **`ece_and_coverage.png`** is **coverage only** (nominal τ line). **`qr_worst_winter_week.png`** and **`cqr_pred_comparison_7d.png`** use the **same worst winter week** (168 contiguous hours maximizing mean shortfall on QR). **`qr_pred_vs_actual_spring_week.png`** — scatter for a **spring** (Mar–May) 168h window with empirical coverage **within ±0.5pp of 95%** when possible (else closest in spring). Holiday **sharpness** → **`qr_holiday_sharpness.png`**. Temperature: **`qr_failure_by_temperature.png`**, **`qr_sharpness_by_temperature.png`**.

In [ ]:
from pathlib import Path
import os

REPO = Path("/zfsauton2/home/yixiz/pjm-load-prediction")
if (Path.cwd() / "plot_progress_presentation.py").is_file():
    REPO = Path.cwd().resolve()
else:
    os.chdir(REPO)
print("cwd:", Path.cwd())

In [ ]:
%matplotlib inline

from IPython.display import Image, display

from plot_progress_presentation import (
    DEFAULT_RUNS,
    DEFAULT_STATIONS,
    build_all_figures,
    load_test_frame,
    load_predictions_csv,
    align_actual_pred,
    compute_ols_mean_forecast,
    plot_timeseries_actual_pred_fill,
    plot_seasonal_failure_rates,
)

RESULTS = REPO / "results"
FIG_DIR = REPO / "results" / "figures_progress"

## Regenerate all PNGs

Equivalent to: `python plot_progress_presentation.py` from the repo root.

In [ ]:
build_all_figures(
    out_dir=FIG_DIR,
    results_dir=RESULTS,
    stations=list(DEFAULT_STATIONS),
    qr_json_name="qr_lr5.0.json",
    run_labels=dict(DEFAULT_RUNS),
    promised_q=0.95,
)

## Figures (saved under `results/figures_progress/`)

In [ ]:
for path in sorted(FIG_DIR.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

## Optional: another run on the same axes

Load any `<name>_predictions.csv` + metrics JSON and reuse `plot_timeseries_actual_pred_fill` (red = actual > pred, i.e. conservative shortfall).

In [ ]:
import matplotlib.pyplot as plt

df_test, feats = load_test_frame(
    list(DEFAULT_STATIONS), RESULTS, "qr_lr5.0.json"
)
mean_ols = compute_ols_mean_forecast(list(DEFAULT_STATIONS), feats, RESULTS)

run = "winter0.25"
pred = load_predictions_csv(RESULTS / f"{run}_predictions.csv")
d = align_actual_pred(df_test, pred)

fig, ax = plt.subplots(figsize=(12, 4))
plot_timeseries_actual_pred_fill(
    d,
    mean_pred=mean_ols,
    resample="7D",
    title=f"{run}: CQR-adjusted τ=0.95 vs actual (7-day mean)",
    ax=ax,
)
fig.tight_layout()
plt.show()